# Tutorial 7: Risk Management -- VaR and CVaR

This notebook demonstrates classical and quantum approaches to Value-at-Risk (VaR)
and Conditional Value-at-Risk (CVaR / Expected Shortfall).

**Reference**: Woerner & Egger (2019) -- Quantum Risk Analysis.

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Generate Return Data

In [ ]:
from qufin.data.synthetic import gbm_paths

# Simulate 1 year of daily prices
paths = gbm_paths(s0=100, mu=0.08, sigma=0.2, T=1.0, n_steps=252, n_paths=1)
prices = paths[0]  # Single path

# Compute log returns
returns = np.diff(np.log(prices))
print(f"Generated {len(returns)} daily returns")
print(f"Mean:  {returns.mean():.6f}")
print(f"Std:   {returns.std():.6f}")
print(f"Skew:  {float(np.mean((returns - returns.mean())**3) / returns.std()**3):.4f}")

## 2. Classical Historical VaR

In [ ]:
from qufin.risk.classical_var import historical_var, parametric_var, monte_carlo_var

# Historical VaR at 99% confidence
hist_result = historical_var(returns, confidence=0.99)

print(f"Historical VaR (99%): {hist_result.var:.6f}")
print(f"Historical ES  (99%): {hist_result.expected_shortfall:.6f}")

## 3. Parametric VaR (Gaussian)

In [ ]:
param_result = parametric_var(returns, confidence=0.99)

print(f"Parametric VaR (99%): {param_result.var:.6f}")
print(f"Parametric ES  (99%): {param_result.expected_shortfall:.6f}")

## 4. Monte Carlo VaR

In [ ]:
mc_result = monte_carlo_var(returns, confidence=0.99, n_simulations=10_000)

print(f"Monte Carlo VaR (99%): {mc_result.var:.6f}")
print(f"Monte Carlo ES  (99%): {mc_result.expected_shortfall:.6f}")

## 5. Quantum VaR

Quantum VaR encodes the loss distribution into quantum amplitudes and uses
amplitude estimation to compute tail probabilities.

In [ ]:
from qufin.risk.quantum_var import quantum_var, QuantumVaRConfig
from qufin.backends.qiskit_backend import QiskitAerBackend

backend = QiskitAerBackend(shots=4096)

q_result = quantum_var(
    returns=returns,
    confidence=0.99,
    backend=backend,
    config=QuantumVaRConfig(n_qubits=5),
)

print(f"Quantum VaR (99%): {q_result.value:.6f}")

## 6. Comparison Table

In [ ]:
print(f"{'Method':<20} {'VaR (99%)':>12} {'ES (99%)':>12}")
print("-" * 46)
print(f"{'Historical':<20} {hist_result.var:>12.6f} {hist_result.expected_shortfall:>12.6f}")
print(f"{'Parametric':<20} {param_result.var:>12.6f} {param_result.expected_shortfall:>12.6f}")
print(f"{'Monte Carlo':<20} {mc_result.var:>12.6f} {mc_result.expected_shortfall:>12.6f}")
print(f"{'Quantum':<20} {q_result.value:>12.6f} {'--':>12}")

## 7. Stress Testing

In [ ]:
from qufin.risk.stress import stress_test

# Apply a -3 sigma shock
stressed = stress_test(
    returns=returns,
    shock=-3 * returns.std(),
)

stressed_var = historical_var(stressed, confidence=0.99)
print(f"Normal VaR:   {hist_result.var:.6f}")
print(f"Stressed VaR: {stressed_var.var:.6f}")

## Summary

In this tutorial we covered:
- Three classical VaR methods (historical, parametric, Monte Carlo)
- Quantum VaR via amplitude estimation
- Stress testing with return shocks

**Next**: Tutorial 08 covers noise models and error mitigation for near-term hardware.